# Convert source json files to the compressed binary format

In [1]:
from glob import glob
from json import loads
import numpy as np
import pandas as pd
import zstandard as zstd

from os import makedirs
from os.path import basename
from simple.utils import pmap

In [2]:
folder = '../json'
opt_folder = '../data/opt'
fut_folder = '../data/fut'

In [3]:
TOrderlog = np.dtype([
    ('ts_event', 'M8[ns]'),
    ('ts_recv', 'M8[ns]'),
    ('action', 'S1'),
    ('side', 'S1'),
    ('price', 'f8'),
    ('size', 'i8'),
    ('channel_id', 'i4'),
    ('order_id', 'u8'),
    ('flags', 'u1'),
    ('ts_in_delta', 'i4'),
    ('sequence', 'i4'),
    ('symbol', 'S45'),
    ('rtype', 'i4'),
    ('publisher_id', 'u4'),
    ('instrument_id', 'i4')
])

TOrderlog.itemsize

112

In [4]:
def iter_orderlog(fname):
    with open(fname, 'rt') as f:
        for s in f:
            j = loads(s)
            hd = j['hd']
            price = j.get('price')
            order_id = j.get('order_id')
            yield (
                np.datetime64(hd['ts_event'][:-1], 'ns'),
                np.datetime64(j['ts_recv'][:-1], 'ns'),
                j['action'],
                j.get('side') or '',
                float(price) if price is not None else np.nan,
                j.get('size') or 0,
                j.get('channel_id') or 0,
                int(order_id) if order_id is not None else 0,
                j.get('flags') or 0,
                j.get('ts_in_delta') or 0,
                j.get('sequence') or 0,
                j.get('symbol') or '',
                hd.get('rtype') or 0,
                hd.get('publisher_id') or 0,
                hd.get('instrument_id') or 0,
            )

In [5]:
def convert(fname, out_folder):
    """repack json file to the zstd-compressed binary formart"""

    L = np.fromiter(iter_orderlog(fname), dtype=TOrderlog)
    makedirs(out_folder, exist_ok=True)
    out_fname = f"{out_folder}/{basename(fname).replace('.json', '.zst')}"

    with open(out_fname, 'wb') as f:
        with zstd.ZstdCompressor(level=18, threads=-1).stream_writer(f) as w:
            w.write(L.tobytes())
            
    return len(L)

In [6]:
def read(fname, dtype=TOrderlog):
    with open(fname, 'rb') as f:
        r = zstd.ZstdDecompressor().stream_reader(f)
        buf = r.readall()
        return np.frombuffer(buf, dtype=dtype)

## Parallel folder conversion

In [7]:
fnames = sorted(glob(f'{folder}/XEUR-20260409-HJTR7RCAKT/xeur-eobi-*.mbo.json'))
fnames

['../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260309.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260310.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260311.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260312.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260313.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260316.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260317.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260318.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260319.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260320.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260323.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260324.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260325.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260326.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260327.mbo.json',
 '../json/XEUR-20260409-H

In [8]:
print(pmap(convert, fnames, opt_folder))

  0%|          | 0/22 [00:00<?, ?it/s]

[1305607, 749817, 780972, 576394, 625371, 451031, 535900, 688888, 1475162, 742003, 781817, 1006619, 818761, 599945, 477378, 475298, 560527, 796538, 577506, 7616, 507846, 634621]


In [9]:
fnames = sorted(glob(f'{opt_folder}/*.zst'))
fnames

['../data/opt/xeur-eobi-20260309.mbo.zst',
 '../data/opt/xeur-eobi-20260310.mbo.zst',
 '../data/opt/xeur-eobi-20260311.mbo.zst',
 '../data/opt/xeur-eobi-20260312.mbo.zst',
 '../data/opt/xeur-eobi-20260313.mbo.zst',
 '../data/opt/xeur-eobi-20260316.mbo.zst',
 '../data/opt/xeur-eobi-20260317.mbo.zst',
 '../data/opt/xeur-eobi-20260318.mbo.zst',
 '../data/opt/xeur-eobi-20260319.mbo.zst',
 '../data/opt/xeur-eobi-20260320.mbo.zst',
 '../data/opt/xeur-eobi-20260323.mbo.zst',
 '../data/opt/xeur-eobi-20260324.mbo.zst',
 '../data/opt/xeur-eobi-20260325.mbo.zst',
 '../data/opt/xeur-eobi-20260326.mbo.zst',
 '../data/opt/xeur-eobi-20260327.mbo.zst',
 '../data/opt/xeur-eobi-20260330.mbo.zst',
 '../data/opt/xeur-eobi-20260331.mbo.zst',
 '../data/opt/xeur-eobi-20260401.mbo.zst',
 '../data/opt/xeur-eobi-20260402.mbo.zst',
 '../data/opt/xeur-eobi-20260406.mbo.zst',
 '../data/opt/xeur-eobi-20260407.mbo.zst',
 '../data/opt/xeur-eobi-20260408.mbo.zst']

In [10]:
Opt = pd.DataFrame(np.concatenate(pmap(read, fnames))).set_index('ts_event')
Opt

  0%|          | 0/22 [00:00<?, ?it/s]

,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
ts_event,,,,,,,,,,,,,,
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'R',b'N',NaN,0,79,0,8,0,0,b'EUCO SI 20260710 PS EU P 1.1650 0',160,101,34513
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'A',b'B',0.02120,20,79,10996414798222631105,0,2365,52012,b'EUCO SI 20260710 PS EU P 1.1650 0',160,101,34513
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'A',b'A',0.02163,20,79,1773042761367855297,128,2365,52012,b'EUCO SI 20260710 PS EU P 1.1650 0',160,101,34513
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'R',b'N',NaN,0,79,0,8,0,0,b'EUCO SI 20260710 PS EU C 1.1650 0',160,101,34512
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'A',b'B',0.01920,20,79,10996414798222631105,0,2365,52012,b'EUCO SI 20260710 PS EU C 1.1650 0',160,101,34512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-08 16:31:31.562120667,2026-04-08 16:31:31.562196101,b'C',b'A',0.00235,20,79,1775665507042983358,128,2444,199054,b'EUCO SI 20260612 PS EU C 1.2125 0',160,101,33321
2026-04-08 16:31:31.562120667,2026-04-08 16:31:31.562196544,b'C',b'B',0.00564,20,79,10999037814277244271,0,1972,199055,b'EUCO SI 20260612 PS EU P 1.1525 0',160,101,33274
2026-04-08 16:31:31.562120667,2026-04-08 16:31:31.562196544,b'C',b'A',0.00609,20,79,1775665777422468463,128,1972,199055,b'EUCO SI 20260612 PS EU P 1.1525 0',160,101,33274


In [11]:
fnames = sorted(glob(f'{folder}/XEUR-20260409-HTT6HHLT6R/xeur-eobi-*.mbo.json'))
fnames

['../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260309.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260310.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260311.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260312.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260313.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260316.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260317.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260318.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260319.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260320.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260323.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260324.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260325.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260326.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260327.mbo.json',
 '../json/XEUR-20260409-H

## Futures (base asset)

In [12]:
print(pmap(convert, fnames, fut_folder))

  0%|          | 0/22 [00:00<?, ?it/s]

[2232542, 1328747, 1022795, 1032880, 862160, 1645242, 555003, 467447, 910536, 635642, 1340785, 1285000, 1107485, 730748, 599680, 527122, 453503, 708364, 723982, 205, 586606, 1113244]


In [13]:
fnames = sorted(glob(f'{fut_folder}/*.zst'))
fnames

['../data/fut/xeur-eobi-20260309.mbo.zst',
 '../data/fut/xeur-eobi-20260310.mbo.zst',
 '../data/fut/xeur-eobi-20260311.mbo.zst',
 '../data/fut/xeur-eobi-20260312.mbo.zst',
 '../data/fut/xeur-eobi-20260313.mbo.zst',
 '../data/fut/xeur-eobi-20260316.mbo.zst',
 '../data/fut/xeur-eobi-20260317.mbo.zst',
 '../data/fut/xeur-eobi-20260318.mbo.zst',
 '../data/fut/xeur-eobi-20260319.mbo.zst',
 '../data/fut/xeur-eobi-20260320.mbo.zst',
 '../data/fut/xeur-eobi-20260323.mbo.zst',
 '../data/fut/xeur-eobi-20260324.mbo.zst',
 '../data/fut/xeur-eobi-20260325.mbo.zst',
 '../data/fut/xeur-eobi-20260326.mbo.zst',
 '../data/fut/xeur-eobi-20260327.mbo.zst',
 '../data/fut/xeur-eobi-20260330.mbo.zst',
 '../data/fut/xeur-eobi-20260331.mbo.zst',
 '../data/fut/xeur-eobi-20260401.mbo.zst',
 '../data/fut/xeur-eobi-20260402.mbo.zst',
 '../data/fut/xeur-eobi-20260406.mbo.zst',
 '../data/fut/xeur-eobi-20260407.mbo.zst',
 '../data/fut/xeur-eobi-20260408.mbo.zst']

In [14]:
Fut = pd.DataFrame(np.concatenate(pmap(read, fnames)))
Fut

  0%|          | 0/22 [00:00<?, ?it/s]

,ts_event,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
0,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'R',b'N',NaN,0,23,0,8,0,0,b'FCEU SI 20260316 PS',160,101,442
1,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'A',b'B',1.15280,20,23,10996386599372464268,0,2634,61463,b'FCEU SI 20260316 PS',160,101,442
2,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'A',b'A',1.15320,20,23,1773014562391096283,128,2634,61463,b'FCEU SI 20260316 PS',160,101,442
3,2026-03-09 00:03:01.430619997,2026-03-09 00:03:01.430638368,b'C',b'A',1.15320,20,23,1773014562391096283,128,1201,61595,b'FCEU SI 20260316 PS',160,101,442
4,2026-03-09 00:03:01.473439135,2026-03-09 00:03:01.473465999,b'A',b'A',1.15330,20,23,1773014581473445312,128,959,61600,b'FCEU SI 20260316 PS',160,101,442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19869713,2026-04-08 20:29:59.286116805,2026-04-08 20:29:59.286131985,b'A',b'A',1.16998,30,23,1775680199286122576,128,995,3575271,b'FCEU SI 20260615 PS',160,101,436
19869714,2026-04-08 20:29:59.319695177,2026-04-08 20:29:59.319717737,b'C',b'B',1.16988,30,23,10999052236140898384,0,1226,3575272,b'FCEU SI 20260615 PS',160,101,436
19869715,2026-04-08 20:29:59.319695177,2026-04-08 20:29:59.319717737,b'A',b'B',1.16987,30,23,10999052236174481524,0,1226,3575272,b'FCEU SI 20260615 PS',160,101,436
19869716,2026-04-08 20:29:59.319695177,2026-04-08 20:29:59.319717737,b'C',b'A',1.16998,30,23,1775680199286122576,0,1226,3575272,b'FCEU SI 20260615 PS',160,101,436


In [25]:
Fut.symbol.value_counts()

b'FCEU SI 20260615 PS'       12486196
b'FCEU SI 20260316 PS'        5731883
b'FCEU SI 20260914 PS'        1639955
b'FCEU.S.MAR26.JUN26.SPD'        3891
b'FCEU.S.APR26.MAY26.SPD'        2220
                               ...   
b'FCEU.S.JUL26.OCT26.SPD'           1
b'FCEU.S.AUG26.OCT26.SPD'           1
b'FCEU.S.SEP26.OCT26.SPD'           1
b'FCEU.S.APR26.NOV26.SPD'           1
b'FCEU.S.DEC26.JAN27.SPD'           1
Name: symbol, Length: 69, dtype: int64

In [24]:
Fut.instrument_id.value_counts()

444       6653860
442       5731891
436       3445548
445       2386804
447        974516
           ...   
659135          1
659136          1
659137          1
659138          1
659120          1
Name: instrument_id, Length: 150, dtype: int64

In [28]:
Fut[Fut.symbol == b'FCEU SI 20260615 PS'].instrument_id.value_counts()

444    6653852
436    3445548
445    2386796
Name: instrument_id, dtype: int64

In [35]:
Fut[Fut.instrument_id == 442]

,ts_event,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
0,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'R',b'N',NaN,0,23,0,8,0,0,b'FCEU SI 20260316 PS',160,101,442
1,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'A',b'B',1.1528,20,23,10996386599372464268,0,2634,61463,b'FCEU SI 20260316 PS',160,101,442
2,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'A',b'A',1.1532,20,23,1773014562391096283,128,2634,61463,b'FCEU SI 20260316 PS',160,101,442
3,2026-03-09 00:03:01.430619997,2026-03-09 00:03:01.430638368,b'C',b'A',1.1532,20,23,1773014562391096283,128,1201,61595,b'FCEU SI 20260316 PS',160,101,442
4,2026-03-09 00:03:01.473439135,2026-03-09 00:03:01.473465999,b'A',b'A',1.1533,20,23,1773014581473445312,128,959,61600,b'FCEU SI 20260316 PS',160,101,442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18169731,2026-04-06 18:53:08.614345509,2026-04-06 18:53:08.614351916,b'R',b'N',NaN,0,23,0,128,6407,3,b'FCEU SI 20261214 PS',160,101,442
18169754,2026-04-06 18:53:08.742343212,2026-04-06 18:53:08.742345380,b'R',b'N',NaN,0,23,0,128,2168,3,b'FCEU SI 20261214 PS',160,101,442
18169771,2026-04-06 18:53:08.742343212,2026-04-06 18:53:08.742357229,b'R',b'N',NaN,0,23,0,128,14017,4,b'FCEU SI 20261214 PS',160,101,442
18169794,2026-04-06 18:53:08.870339845,2026-04-06 18:53:08.870340934,b'R',b'N',NaN,0,23,0,128,1089,4,b'FCEU SI 20261214 PS',160,101,442


In [21]:
Opt.symbol.value_counts()

b'EUCO SI 20260410 PS EU C 1.1575 0'    156253
b'EUCO SI 20260410 PS EU C 1.1600 0'    155052
b'EUCO SI 20260612 PS EU C 1.1575 0'    149925
b'EUCO SI 20260410 PS EU C 1.1625 0'    147576
b'EUCO SI 20260710 PS EU C 1.1600 0'    146774
                                         ...  
b'EUCO SI 20261211 PS EU C 1.1100 0'         8
b'EUCO SI 20260410 PS EU C 1.1000 0'         8
b'EUCO.O.260310.PDIA.000001'                 3
b'EUCO SI 20260313 PS EU P 1.1900 0'         2
b'EUCO.O.260331.RBUL.000001'                 1
Name: symbol, Length: 987, dtype: int64

In [37]:
np.save('/tmp/FCEU SI 20260316 PS.npy', f)

In [44]:
F = pd.DataFrame(f).set_index('ts_event')
F.action.value_counts()

b'C'    1015113
b'A'    1015090
b'T'         44
b'F'         44
b'R'          1
b'M'          1
Name: action, dtype: int64

In [51]:
F[F.order_id == 10996414336265404691]

,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
ts_event,,,,,,,,,,,,,,
2026-03-09 07:44:59.410622271,2026-03-09 07:44:59.410637352,b'A',b'B',1.1561,20,23,10996414336265404691,128,651,1569926,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:45:01.731911761,2026-03-09 07:45:01.735130301,b'F',b'B',1.1561,18,23,10996414336265404691,0,1304,1570115,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:45:01.731911761,2026-03-09 07:45:01.735130301,b'C',b'B',1.1561,18,23,10996414336265404691,128,1304,1570115,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:45:01.738677045,2026-03-09 07:45:01.738691799,b'C',b'B',1.1561,2,23,10996414336265404691,128,840,1570116,b'FCEU SI 20260316 PS',160,101,442


In [46]:
F.loc['2026-03-09 07:45:00':]

/tmp/ipykernel_758891/1652679406.py:1: FutureWarning: Value based partial slicing on non-monotonic DatetimeIndexes with non-existing keys is deprecated and will raise a KeyError in a future Version.
  F.loc['2026-03-09 07:45:00':]


,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
ts_event,,,,,,,,,,,,,,
2026-03-09 07:45:00.089199499,2026-03-09 07:45:00.089214201,b'C',b'B',1.15608,20,23,10996414336268097186,0,708,1570030,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:45:00.089199499,2026-03-09 07:45:00.089214201,b'A',b'B',1.15607,20,23,10996414336943981373,128,708,1570030,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:45:01.089804109,2026-03-09 07:45:01.089820179,b'C',b'B',1.15607,20,23,10996414336943981373,0,805,1570080,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:45:01.089804109,2026-03-09 07:45:01.089820179,b'A',b'B',1.15608,20,23,10996414337944586500,128,805,1570080,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:45:01.731911761,2026-03-09 07:45:01.735130301,b'T',b'A',1.15610,18,23,1773042301735051569,0,1304,1570115,b'FCEU SI 20260316 PS',160,101,442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-09 20:59:56.129395311,2026-03-09 20:59:56.129410971,b'A',b'A',1.16410,20,23,1773089996129401590,128,633,6413045,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 20:59:56.129549359,2026-03-09 20:59:56.129560908,b'C',b'A',1.16420,20,23,1773089995088358222,128,803,6413047,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 20:59:56.137225481,2026-03-09 20:59:56.137241200,b'A',b'B',1.16350,20,23,10996462032992007174,128,651,6413053,b'FCEU SI 20260316 PS',160,101,442


In [50]:
F[F.action == b'F']

,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
ts_event,,,,,,,,,,,,,,
2026-03-09 07:45:01.731911761,2026-03-09 07:45:01.735130301,b'F',b'B',1.15610,18,23,10996414336265404691,0,1304,1570115,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:55:56.902588117,2026-03-09 07:55:56.905837606,b'F',b'B',1.15659,20,23,10996414993180550785,0,1060,1630889,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 07:57:12.051081355,2026-03-09 07:57:12.054193272,b'F',b'B',1.15684,8,23,10996415064995815673,0,1236,1638510,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 08:17:51.752653363,2026-03-09 08:17:51.755881912,b'F',b'A',1.15502,1,23,1773044271691369508,0,952,1839685,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 08:53:00.942816397,2026-03-09 08:53:00.945944453,b'F',b'B',1.15267,20,23,10996418417800398348,0,6718,2152154,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 09:08:41.353898379,2026-03-09 09:08:41.357018728,b'F',b'B',1.15400,10,23,10996419358205983270,0,1508,2281500,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 09:31:26.829000257,2026-03-09 09:31:26.832075421,b'F',b'A',1.15550,20,23,1773048686827987917,0,1023,2386458,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 09:32:43.590629227,2026-03-09 09:32:43.593715260,b'F',b'A',1.15636,20,23,1773048763553772425,0,1661,2395855,b'FCEU SI 20260316 PS',160,101,442
2026-03-09 09:32:43.602194815,2026-03-09 09:32:43.605257455,b'F',b'B',1.15646,20,23,10996420800459609994,0,1411,2395885,b'FCEU SI 20260316 PS',160,101,442
